# 🌿 BioEcoInt — Pipeline Completo de Treinamento, SOTA Benchmark & Comitê Temporal
> **Laboratório IFMaker / BioEcoInt**  
> **Arquiteturas Avaliadas:** ResNet-18, MobileNetV4-Conv-Small e RepViT-M0.9  
> **Estratégias Avaliadas:** Baseline Sanitizado, YOLO Crops, SAM 2 Crops e Coordinate Attention (CA)  
> **Estudo de Funções de Perda:** Cross-Entropy (4A), Focal Loss (4B) e Balanced Softmax (4C) no Campeão RepViT-M0.9


### 1. Verificação de Aceleração por Hardware (GPU)


In [ ]:
import torch
print(f"CUDA disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Dispositivo GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memória VRAM Total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


### 2. Instalação do Gerenciador de Pacotes Ultrarrápido (`uv`)


In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = f"/root/.local/bin:{os.environ['PATH']}"
!uv --version


### 3. Clonagem do Repositório (`bio-interaction-ml`)


In [ ]:
import os
from google.colab import userdata

try:
    github_token = userdata.get("GITHUB_TOKEN")
    repo_url = f"https://{github_token}@github.com/Emmerson2018/bio-interaction-ml.git"
except Exception:
    repo_url = "https://github.com/Emmerson2018/bio-interaction-ml.git"

if not os.path.exists("bio-interaction-ml"):
    !git clone {repo_url}
    %cd bio-interaction-ml
else:
    %cd bio-interaction-ml
    !git pull origin main


### 4. Sincronização do Ambiente Virtual (`uv sync`)


In [ ]:
!uv sync
!uv run python -c "import torch, torchvision, ultralytics; print('✅ Ambiente pronto!')"


### 5. Montagem do Google Drive & Sincronização dos Datasets via DVC


In [ ]:
from google.colab import drive, files
import os, shutil, subprocess, json

# 1. Montagem do Google Drive
drive.mount("/content/drive")

# 2. Upload Obrigatório das Credenciais Headless do DVC
print("📤 Faça o upload do arquivo 'default.json' para autenticar o DVC:")
uploaded = files.upload()

default_json_file = None
for fname in uploaded:
    if fname.endswith(".json"):
        default_json_file = fname
        break

if not default_json_file or not os.path.exists(default_json_file):
    raise RuntimeError("❌ Falha crítica: O arquivo 'default.json' é obrigatório para autenticar o DVC.")

with open(default_json_file, "r") as f:
    creds = json.load(f)

cid = creds.get("client_id")
cs = creds.get("client_secret")

if not cid or not cs:
    raise ValueError("❌ Falha crítica: 'default.json' inválido. Campos 'client_id' ou 'client_secret' ausentes.")

creds_dir = f"/root/.cache/pydrive2fs/{cid}"
os.makedirs(creds_dir, exist_ok=True)
target_file = os.path.join(creds_dir, "default.json")
shutil.move(default_json_file, target_file)
print(f"✅ Credenciais instaladas com sucesso em {target_file}")

# Configura o segredo local no remote do DVC
subprocess.run(["uv", "run", "dvc", "remote", "modify", "--local", "gdrive-storage", "gdrive_client_secret", cs], check=True)

# 3. Download e Descompactação Direta dos Datasets via DVC
LOCAL_DATA_DIR = "datasets/generated"
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

for ds_name in ["animals_video_sanitized", "animals_video_sam2_crops"]:
    zip_name = f"{ds_name}.zip"
    target_dir = os.path.join(LOCAL_DATA_DIR, ds_name)
    local_zip = os.path.join(LOCAL_DATA_DIR, zip_name)
    dvc_file = f"{LOCAL_DATA_DIR}/{zip_name}.dvc"
    
    if os.path.exists(target_dir) and len(os.listdir(target_dir)) > 0:
        print(f"✅ Dataset {ds_name} já presente em {target_dir}!")
        continue

    print(f"⬇️ Puxando {zip_name} via DVC...")
    res = subprocess.run(["uv", "run", "dvc", "pull", dvc_file], check=False)
    if res.returncode != 0 or not os.path.exists(local_zip):
        raise RuntimeError(f"❌ Falha ao baixar {zip_name} via DVC. Verifique as credenciais ou o remote gdrive-storage.")

    print(f"📂 Descompactando {zip_name}...")
    subprocess.run(["unzip", "-q", "-o", local_zip, "-d", LOCAL_DATA_DIR], check=True)
    print(f"✅ {ds_name} pronto com sucesso!")


---
## 📦 FAMÍLIA 1: ResNet-18 (Baseline Clássico)
Avaliação de controle com convoluções residuais clássicas nas 4 estratégias de visão.


#### Exp 1.1: ResNet-18 Baseline Sanitizado (Cross-Entropy)


In [ ]:
!MPLBACKEND=Agg uv run base-train -opt options/exp1_resnet18_baseline.yml


#### Exp 1.2: ResNet-18 + YOLO Crops (Two-Stage)


In [ ]:
!MPLBACKEND=Agg uv run base-train -opt options/exp2_resnet18_yolo_crop.yml


#### Exp 1.3: ResNet-18 + Coordinate Attention (CA)


In [ ]:
!MPLBACKEND=Agg uv run base-train -opt options/exp3_resnet18_ca.yml


#### Exp 1.4: ResNet-18 + SAM 2 Crops (Segmentação por Instância)


In [ ]:
!MPLBACKEND=Agg uv run base-train -opt options/exp_resnet18_sam2_crops.yml


---
## ⚡ FAMÍLIA 2: Google MobileNetV4-Small (Universal Inverted Bottlenecks - UIB)
Avaliação da arquitetura móvel convolucional moderna (3.8M parâmetros) nas 4 estratégias de visão.


#### Exp 2.1: MobileNetV4-Small Baseline (Sanitizado + Cross-Entropy)


In [ ]:
!MPLBACKEND=Agg uv run base-train -opt options/exp_mobilenetv4_small_baseline.yml


#### Exp 2.2: MobileNetV4-Small + YOLO Crops (Two-Stage)


In [ ]:
!MPLBACKEND=Agg uv run base-train -opt options/exp_mobilenetv4_small_yolo_crop.yml


#### Exp 2.3: MobileNetV4-Small + Coordinate Attention (CA)


In [ ]:
!MPLBACKEND=Agg uv run base-train -opt options/exp_mobilenetv4_small_ca.yml


#### Exp 2.4: MobileNetV4-Small + SAM 2 Crops (Segmentação por Instância)


In [ ]:
!MPLBACKEND=Agg uv run base-train -opt options/exp_mobilenetv4_sam2_crops.yml


---
## 🚀 FAMÍLIA 3: RepViT-M0.9 (Vision Transformer Reparametrizável)
Avaliação da arquitetura campeã (4.7M parâmetros) combinando atenção global de ViT com reparametrização estrutural mobile.


#### Exp 3.1: RepViT-M0.9 Baseline (Sanitizado + Cross-Entropy / Exp 4A)


In [ ]:
!MPLBACKEND=Agg uv run base-train -opt options/exp_repvit_m0_9_baseline.yml


#### Exp 3.2: RepViT-M0.9 + YOLO Crops (Two-Stage)


In [ ]:
!MPLBACKEND=Agg uv run base-train -opt options/exp_repvit_m0_9_yolo_crop.yml


#### Exp 3.3: RepViT-M0.9 + Coordinate Attention (CA)


In [ ]:
!MPLBACKEND=Agg uv run base-train -opt options/exp_repvit_m0_9_ca.yml


#### Exp 3.4: RepViT-M0.9 + SAM 2 Crops (Recortes Limpos Exemplar V2)


In [ ]:
!MPLBACKEND=Agg uv run base-train -opt options/exp_repvit_m0_9_sam2_crops.yml


---
## 🔬 ESTUDO DE FUNÇÕES DE PERDA NO REPVIT-M0.9 (Exp 4A, 4B e 4C)
Análise iso-condicional do impacto do desbalanceamento severo na modelagem de dependências não-locais de longo alcance do Transformer.


#### Exp 4A: RepViT-M0.9 + Cross-Entropy (Equivalente ao Baseline)


In [ ]:
!MPLBACKEND=Agg uv run base-train -opt options/exp_loss_ce_repvit.yml


#### Exp 4B: RepViT-M0.9 + Focal Loss (gamma = 2.0)


In [ ]:
!MPLBACKEND=Agg uv run base-train -opt options/exp_loss_focal_repvit.yml


#### Exp 4C: RepViT-M0.9 + Balanced Softmax (Recorde SOTA 80.26% Macro-F1)


In [ ]:
!MPLBACKEND=Agg uv run base-train -opt options/exp_repvit_m0_9.yml


---
## 📹 AVALIAÇÃO ESTÉRIL & COMITÊ TEMPORAL EM VÍDEO (2s @ 3 FPS)
Executa teste estéril e simula o comitê temporal de inferência contínua com consenso >= 70% e abstenção protetora.


In [ ]:
# 1. Avaliação Estéril do Campeão RepViT SAM 2 Crops
!uv run base-test -opt options/exp_repvit_m0_9_sam2_crops.yml --checkpoint experiments/Exp_RepViT_M0_9_SAM2_Crops/models/best_macro_f1.pth

# 2. Comitê Temporal nos Vídeos de Teste do RepViT SAM 2 Crops
!uv run base-test-committee -opt options/exp_repvit_m0_9_sam2_crops.yml --checkpoint experiments/Exp_RepViT_M0_9_SAM2_Crops/models/best_macro_f1.pth --window-sec 2.0 --target-fps 3.0 --consensus-thresh 0.70 --output-report experiments/Exp_RepViT_M0_9_SAM2_Crops/committee_sam2_report.json


---
## ⚙️ OTIMIZAÇÃO DE HIPERPARÂMETROS HPO (Optuna + ASHA Pruning)
Busca bayesiana TPE com poda precoce focada no RepViT-M0.9.


In [ ]:
!uv run python experiments/hpo_optuna_baseline.py -opt options/exp_repvit_m0_9.yml --trials 15 --epochs-per-trial 8


---
## 📱 EXPORTAÇÃO MOBILE E QUANTIZAÇÃO (TFLite FP16 & INT8 PTQ)
Converte os modelos campeões para TFLite FP32, FP16 e INT8 calibrado (PTQ) via reparametrização estrutural `switch_to_deploy`.


In [ ]:
# Exportação do Campeão RepViT SAM 2 Crops
!MPLBACKEND=Agg uv run base-export -opt options/exp_repvit_m0_9_sam2_crops.yml --checkpoint experiments/Exp_RepViT_M0_9_SAM2_Crops/models/best_macro_f1.pth --format all

# Exportação do Campeão Geral RepViT Balanced Softmax
!MPLBACKEND=Agg uv run base-export -opt options/exp_repvit_m0_9.yml --checkpoint experiments/Exp_RepViT_M0_9_BalancedSoftmax/models/best_macro_f1.pth --format all


---
## 📊 CONSOLIDAÇÃO AUTOMÁTICA DE RESULTADOS & GRÁFICOS SOTA
Compila as métricas de validação, teste estéril e comitê temporal de todos os experimentos em tabela e gráfico comparativo.


In [ ]:
!uv run python consolidar_resultados_finais.py


---
## 💾 SINCRONIZAÇÃO DE RESULTADOS & DVC NO GOOGLE DRIVE
Salva os pesos e modelos exportados no Google Drive via DVC e sincroniza o Git com pull preventivo.


In [ ]:
import os, glob, subprocess

# 1. Versionar diretórios de modelos e exportações no DVC
for exp_dir in sorted(glob.glob("experiments/Exp_*")):
    models_dir = os.path.join(exp_dir, "models")
    export_dir = os.path.join(exp_dir, "export")
    if os.path.exists(models_dir) and not os.path.exists(f"{models_dir}.dvc"):
        print(f"📦 Adicionando ao DVC: {models_dir}")
        subprocess.run(["uv", "run", "dvc", "add", models_dir], check=False)
    if os.path.exists(export_dir) and not os.path.exists(f"{export_dir}.dvc"):
        print(f"📦 Adicionando ao DVC: {export_dir}")
        subprocess.run(["uv", "run", "dvc", "add", export_dir], check=False)

# 2. Push dos arquivos binários pesados para o Google Drive
print("☁️ Enviando artefatos pesados para o Google Drive via DVC...")
!uv run dvc push -r gdrive-storage || true

# 3. Git commit e push preventivo com git pull --rebase
!git config user.name "IFMaker Colab Runner"
!git config user.email "colab@ifmaker.edu.br"
!git add experiments/ options/ *.py *.md notebooks/
!git commit -m "feat(colab): consolidacao de novos experimentos e modelos DVC" || echo "Sem alteracoes para commit"

print("🔄 Sincronizando com git pull antes do push...")
!git pull --rebase origin main || true
!git push origin main || echo "⚠️ Push rejeitado. Modelos preservados no Google Drive via DVC!"


---
## 🛑 DESCONEXÃO AUTOMÁTICA DO RUNTIME (Proteção de Compute Units)
Encerra a sessão da GPU após a conclusão para economizar créditos.


In [ ]:
from google.colab import runtime
AUTO_DISCONNECT = False  # Mude para True se desejar desconectar automaticamente ao final do treino noturno
if AUTO_DISCONNECT:
    print("🔌 Desconectando runtime do Google Colab...")
    runtime.unassign()
else:
    print("✅ Sessão mantida ativa para inspeção interativa.")
